In [ ]:
import requests
import time
path_data = '././data/raw/DataCoSupplyChainDataset.csv'
city_coords = {}
count = 0
df = spark.read.csv(path_data, header=True, inferSchema=True)

all_locations = df.select("Customer City","Customer State").distinct().collect()

for row in all_locations:
    city = row["Customer City"]
    state = row["Customer State"]

    try:
        response = requests.get(
            f"https://api.geoapify.com/v1/geocode/search?text={city},{state}&apiKey=dbdc93fb7bed484da2993b88667c8a20",
            timeout=10
        )
        data_city = response.json()
        features = data_city.get("features", [])
        if features:
            city_data = features[0]
            latitude = city_data["properties"].get("lat", 0)
            longitude = city_data["properties"].get("lon", 0)
        else:
            latitude, longitude = 0, 0

    except Exception as e:
        print(f"Error for {city}, {state}: {e}")
        latitude, longitude = 0, 0

    city_coords[city] = (latitude, longitude)
    count += 1
    print(count, latitude, longitude)

    time.sleep(0.1)  

1 40.3496953 -74.6597376
2 40.7720145 -73.9302673
3 33.4151005 -111.831455
4 33.9092802 -118.0849169
5 42.3602273 -87.8318164
Error for Ontario, CA: HTTPSConnectionPool(host='api.geoapify.com', port=443): Max retries exceeded with url: /v1/geocode/search?text=Ontario,CA&apiKey=dbdc93fb7bed484da2993b88667c8a20 (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1017)')))
6 0 0
7 34.0922947 -117.43433
8 39.6828358 -75.7515682
9 33.8714814 -117.8617337
10 37.3541132 -121.955174
11 37.3228934 -122.0322895
12 32.9961038 -80.0387292
13 40.6576071 -73.5835826
14 40.7934789 -74.1979277
15 34.6086854 -98.3903305
16 38.6779591 -121.176058
17 41.6664781 -81.3399769
18 33.0186699 -80.1762704
19 38.8950368 -77.0365427
20 30.5335302 -92.081509
21 32.7947731 -116.962526
22 34.2204227 -118.3878945
23 29.8826436 -97.9405828
24 42.4184296 -71.1061639
25 35.8460396 -86.3921096
26 33.870413 -117.9962165
27 39.3458953 -84.5605031
28 34.0286226 -117.8103367
29 43.597646 -84.766

In [ ]:
coords_list = [(city, float(lat), float(lon)) for city, (lat, lon) in city_coords.items()]
coords_df = spark.createDataFrame(coords_list, ["Customer City", "New_Latitude", "New_Longitude"])

df = df.join(coords_df, on="Customer City", how="left")
output_path = "./data/processed/DataCoSupplyChain_with_coords.csv"

pdf_with_new_columns=df.toPandas()
pdf_with_new_columns.to_csv(output_path, header=True ,index=False)

print(f"DataFrame sauvegardé en CSV à : {output_path}")


25/11/13 11:24:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame sauvegardé en CSV à : ./data/processed/DataCoSupplyChain_with_coords.csv


In [35]:
from pyspark.sql.functions import col


output_path = "./data/processed/DataCoSupplyChain_with_coords.csv"

spark.read.format('csv').option("header","True").load(output_path)

df = df.filter(col("Delivery Status") != "Shipping canceled")


df.filter(col("Delivery Status") == "Shipping canceled").count()



0

In [32]:
from pyspark.sql.functions import col, radians, sin, cos, sqrt, asin

r = 6371  

df = df.withColumn("dlat", radians(col("New_Latitude") - col("Latitude"))) \
       .withColumn("dlon", radians(col("New_Longitude") - col("Longitude"))) \
       .withColumn("lat1_rad", radians(col("Latitude"))) \
       .withColumn("lat2_rad", radians(col("New_Latitude"))) \
       .withColumn(
           "a",
           sin(col("dlat")/2)**2 + cos(col("lat1_rad")) * cos(col("lat2_rad")) * sin(col("dlon")/2)**2
       ) \
       .withColumn("c", 2 * asin(sqrt(col("a")))) \
       .withColumn("distance_km", col("c") * r) \
       .drop("dlat", "dlon", "lat1_rad", "lat2_rad", "a", "c")

df.select("Customer City", "New_Latitude", "New_Longitude", "distance_km").show(10)



+-------------+------------+-------------+-------------------+
|Customer City|New_Latitude|New_Longitude|        distance_km|
+-------------+------------+-------------+-------------------+
|       Caguas|    18.23598|   -66.030975| 1.8365019708395816|
|       Caguas|    18.23598|   -66.030975|  4.876366598033297|
|       Caguas|    18.23598|   -66.030975|  2.079421479159196|
|       Caguas|    18.23598|   -66.030975| 0.9717258094087833|
|       Caguas|    18.23598|   -66.030975| 0.7116376722835362|
|  Los Angeles|  34.0536909|  -118.242766|  9.181145513992359|
|      Salinas|  36.6744117|  -121.655037|0.25161498484303724|
|     San Jose|  37.3361663|  -121.890591|  4.954139531313116|
|     Freeport|  40.6576071|  -73.5835826|  0.423918282758481|
|        Miami|  25.7741566|  -80.1935973|  17.47584538527845|
+-------------+------------+-------------+-------------------+
only showing top 10 rows



In [33]:
df.show()

+--------------+--------+------------------------+-----------------------------+-----------------+------------------+----------------+------------------+-----------+--------------+----------------+--------------+--------------+-----------+--------------+-----------------+----------------+--------------+--------------------+----------------+-------------+---------------+-----------+------------+------------+----------+-------------+-----------------+-----------------------+--------+----------------------+-------------------+------------------------+-------------+------------------------+-----------------------+-------------------+------+----------------+----------------------+--------------+--------------------+---------------+-------------+---------------+-------------------+-------------------+--------------------+------------+-------------+--------------+--------------------------+--------------+------------+-------------+-------------------+
| Customer City|    Type|Days for shippin

In [ ]:
cols_to_use=[
'Type',
 'Late_delivery_risk',
 'Category Name',
 'Customer Segment',
 'Order Item Quantity',
 'Sales',
 'Order Profit Per Order',
 'Order Region',
 'Product Price',
 'order date (DateOrders)',
 'Shipping Mode',
 'distance_km'
 ]
df_clean =df[cols_to_use]
print(len(df_clean.columns))
df_clean.show()


12
+--------+------------------+-----------+----------------+-------------------+------+----------------------+--------------+-------------+-----------------------+--------------+-------------------+
|    Type|Late_delivery_risk|Category Id|Customer Segment|Order Item Quantity| Sales|Order Profit Per Order|  Order Region|Product Price|order date (DateOrders)| Shipping Mode|        distance_km|
+--------+------------------+-----------+----------------+-------------------+------+----------------------+--------------+-------------+-----------------------+--------------+-------------------+
|   DEBIT|                 0|         73|        Consumer|                  1|327.75|                 91.25|Southeast Asia|       327.75|        1/31/2018 22:56|Standard Class| 1.8365019708395816|
|TRANSFER|                 1|         73|        Consumer|                  1|327.75|          -249.0899963|    South Asia|       327.75|        1/13/2018 12:27|Standard Class|  4.876366598033297|
| PAYMENT|  

In [39]:
df_final=df.toPandas()
output_path = "./data/processed/DataCoSupplyChain_with_coords_distance.csv"

df_final.to_csv(output_path,header=True,index=False)

In [40]:
print(spark.sparkContext.uiWebUrl)

http://baa38e4da569:4041
